# 03 · Feature engineering
**African Credit Scoring Challenge**

This notebook:
1. Builds time-safe customer-history, pricing, currency and macro features.
2. Tests whether markup is a price or a leaked outcome.
3. Scores each feature family with the validation protocol from notebook 02, on two tracks: **competition** (markup allowed) and **production** (no fields derived from amounts to be repaid).
4. Measures how far Ghana sits outside the training range.
5. Records the assumptions log.

**Protocol: this notebook is the v1 record.** It uses `src/archive/features_v1.py` and `src/evaluate_v1.py`, the versions that produced its numbers, and tags its log rows `protocol = v1`. The key models are re-confirmed under the corrected nested evaluator (v2) in `03b_confirm_v2.ipynb`; those are the numbers reported.

A full run takes about 15 minutes.

In [1]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

sys.path.insert(0, 'src')
sys.path.insert(0, 'src/archive')
from features_v1 import build_features, FAMILIES, MARKUP_HINGE   # v1 record: the code that produced these results
from evaluate_v1 import Evaluator, log_experiment

warnings.filterwarnings('ignore')
DATA, OUT = Path('.'), Path('outputs')
train = pd.read_csv(DATA / 'Train.csv', parse_dates=['disbursement_date', 'due_date'])
test  = pd.read_csv(DATA / 'Test.csv',  parse_dates=['disbursement_date', 'due_date'])
econ  = pd.read_csv(DATA / 'economic_indicators.csv')
folds = pd.read_csv(OUT / 'folds.csv')          # the fixed notebook 02 folds: never re-split
print(train.shape, test.shape, folds.shape)

(68654, 16) (18594, 15) (68654, 3)


## 1. Build the features
Each function in `src/archive/features_v1.py` has a docstring saying what it may and may not use.

In [2]:
feats = build_features(train, test, econ, label_lag_days=0, macro_lag_years=1)
n = len(train)
F_train, F_test = feats.iloc[:n], feats.iloc[n:]
print(feats.shape)
for fam, cols in FAMILIES.items():
    print(f'{fam:9s}', cols)
feats.drop(columns=['lender_id', 'loan_type']).describe().T.round(3)

(87248, 22)
baseline  ['log_amount_lcu', 'markup', 'duration', 'is_new', 'lender_share']
pricing   ['daily_rate', 'markup_high', 'markup_excess']
history   ['hist_n_prior', 'hist_days_since_last', 'hist_log_amount_vs_prior', 'hist_n_known', 'hist_n_defaults', 'hist_default_rate']
usd       ['log_amount_usd']
macro     ['macro_inflation', 'macro_deposit_rate', 'macro_unemployment']
codes     ['lender_id', 'loan_type']


,count,mean,std,min,25%,50%,75%,max
log_amount_lcu,87248.0,8.561,1.265,1.099,7.723,8.537,9.325,16.951
markup,87248.0,0.023,0.059,-1.000,0.000,0.011,0.030,2.165
duration,87248.0,9.607,20.669,1.000,7.000,7.000,7.000,1096.000
daily_rate,87248.0,0.003,0.007,-0.143,0.000,0.001,0.004,0.309
markup_high,87248.0,0.104,0.306,0.000,0.000,0.000,0.000,1.000
markup_excess,87248.0,0.009,0.053,0.000,0.000,0.000,0.000,2.128
is_new,87248.0,0.008,0.091,0.000,0.000,0.000,0.000,1.000
lender_share,87248.0,0.216,0.128,0.000,0.130,0.300,0.300,1.168
log_amount_usd,87248.0,4.010,1.317,0.014,3.111,3.922,4.774,12.254
macro_deposit_rate,87248.0,6.995,1.062,6.687,6.687,6.687,6.687,12.052


## 2. Customer history: check there is no time travel
The rule: a prior loan's outcome counts only if that loan was disbursed **before** this one and its due date had **passed** by this loan's disbursement date.

In [3]:
h = FAMILIES['history']
y = train['target'].to_numpy()
known = F_train['hist_n_known'].to_numpy()
print(f"Train rows with no known history: {(known == 0).sum():,}, default rate {y[known == 0].mean():.1%}")
print(f"Train rows with known history   : {(known > 0).sum():,}, default rate {y[known > 0].mean():.1%}")
print(f"Train rows with an earlier known default: {(F_train['hist_n_defaults'] > 0).sum():,}")
gh = (test['country_id'] == 'Ghana').to_numpy()
print(f"Ghana rows with any known history: {(F_test['hist_n_known'].to_numpy()[gh] > 0).sum()} of {gh.sum():,}")

Train rows with no known history: 11,990, default rate 4.4%
Train rows with known history   : 56,664, default rate 1.3%
Train rows with an earlier known default: 123
Ghana rows with any known history: 0 of 3,525


In [4]:
# Brute-force audit of the customer-history features.
# Rebuild a loan-level table from the raw train + test rows. A loan's outcome is
# known only for labelled training loans whose due date (+ lag) is before the
# current loan's disbursement date.
pooled = pd.concat([
    train[['customer_id', 'tbl_loan_id', 'disbursement_date', 'due_date',
           'Total_Amount', 'target']],
    test.assign(target=np.nan)[['customer_id', 'tbl_loan_id', 'disbursement_date',
                                'due_date', 'Total_Amount', 'target']]
], ignore_index=True)

loan_rows = (pooled.groupby(['customer_id', 'tbl_loan_id'], as_index=False)
             .agg(disb=('disbursement_date', 'first'),
                  due=('due_date', 'first'),
                  target=('target', 'max')))
rng = np.random.default_rng(42)
sample_loan_ids = (train[['customer_id', 'tbl_loan_id']]
                   .drop_duplicates()
                   .sample(n=min(200, train['tbl_loan_id'].nunique()),
                           random_state=42))

checks = []
for row in sample_loan_ids.itertuples(index=False):
    current = loan_rows[
        (loan_rows.customer_id == row.customer_id) &
        (loan_rows.tbl_loan_id == row.tbl_loan_id)
    ].iloc[0]
    customer_loans = loan_rows[loan_rows.customer_id == row.customer_id]
    prior = customer_loans['disb'] < current['disb']
    known_prior = (prior &
                   (customer_loans['due'] < current['disb']) &
                   customer_loans['target'].notna())

    raw_n_prior = int(prior.sum())
    raw_n_known = int(known_prior.sum())
    feature_row = train[
        (train.customer_id == row.customer_id) &
        (train.tbl_loan_id == row.tbl_loan_id)
    ].index[0]
    checks.append({
        'customer_id': row.customer_id,
        'tbl_loan_id': row.tbl_loan_id,
        'raw_n_prior': raw_n_prior,
        'feature_n_prior': int(F_train.loc[feature_row, 'hist_n_prior']),
        'raw_n_known': raw_n_known,
        'feature_n_known': int(F_train.loc[feature_row, 'hist_n_known'])
    })

history_audit = pd.DataFrame(checks)
assert (history_audit['raw_n_prior'] == history_audit['feature_n_prior']).all(), \
    'hist_n_prior disagrees with the independent raw-data calculation.'
assert (history_audit['raw_n_known'] == history_audit['feature_n_known']).all(), \
    'hist_n_known disagrees with the independent raw-data calculation.'

print(f"Time-safety audit passed for {len(history_audit):,} distinct training loans.")
print(f"Maximum absolute difference: "
      f"prior={((history_audit.raw_n_prior-history_audit.feature_n_prior).abs().max())}, "
      f"known={((history_audit.raw_n_known-history_audit.feature_n_known).abs().max())}.")

Time-safety audit passed for 200 distinct training loans.
Maximum absolute difference: prior=0, known=0.


## 3. Is markup a price, or a leaked outcome?
If `Total_Amount_to_Repay` is fixed at signing, loans of the same product and term should carry similar markups whether or not they later defaulted.

In [5]:
tr = train.assign(markup=F_train['markup'].to_numpy())
grp = tr.groupby(['loan_type', 'duration'])
big = grp.filter(lambda g: len(g) > 300 and g.target.sum() >= 10)
leak_table = (big.groupby(['loan_type', 'duration', 'target'])['markup'].median().unstack()
                 .rename(columns={0: 'median_markup_repaid', 1: 'median_markup_defaulted'}))
leak_table['rows'] = big.groupby(['loan_type', 'duration']).size()
display(leak_table.sort_values('rows', ascending=False).style.format(
    {'median_markup_repaid': '{:.1%}', 'median_markup_defaulted': '{:.1%}', 'rows': '{:,}'}))
above = (tr[tr.target == 1].assign(hi=lambda d: d.markup > MARKUP_HINGE)
           .groupby('loan_type').hi.agg(['size', 'mean']).query('size >= 30'))
print(f"Share of defaulted loans above the {MARKUP_HINGE:.1%} cliff, by product: "
      f"{above['mean'].min():.0%}–{above['mean'].max():.0%}")

,target,median_markup_repaid,median_markup_defaulted,rows
loan_type,duration,,,
Type_1,7,0.7%,41.2%,"61,723"
Type_7,7,3.5%,8.3%,"2,790"
Type_5,14,5.6%,12.3%,"1,466"
Type_4,30,7.0%,20.6%,592
Type_6,7,2.0%,35.6%,352


Share of defaulted loans above the 3.7% cliff, by product: 93%–100%


In [6]:
# Repeat the markup analysis using the lender's own amounts.
# Zero-funded rows cannot have a meaningful lender markup and are excluded.
lender_denom = train['Amount_Funded_By_Lender']
lender_markup = (
    train['Lender_portion_to_be_repaid'] /
    lender_denom.where(lender_denom > 0)
) - 1

lender_tr = train.assign(lender_markup=lender_markup)
lender_valid = lender_tr[lender_tr['lender_markup'].notna()].copy()
lender_groups = lender_valid.groupby(['loan_type', 'duration'])
lender_big = lender_groups.filter(
    lambda g: len(g) > 300 and g['target'].sum() >= 10
)
lender_leak_table = (
    lender_big.groupby(['loan_type', 'duration', 'target'])['lender_markup']
    .median().unstack()
    .rename(columns={0: 'median_lender_markup_repaid',
                     1: 'median_lender_markup_defaulted'})
)
lender_leak_table['rows'] = lender_big.groupby(
    ['loan_type', 'duration']
).size()
display(lender_leak_table.sort_values('rows', ascending=False).style.format(
    {'median_lender_markup_repaid': '{:.1%}',
     'median_lender_markup_defaulted': '{:.1%}', 'rows': '{:,}'}
))

lender_defaulted = lender_valid[lender_valid['target'] == 1]
lender_above_share = (
    lender_defaulted.assign(above_hinge=lambda d: d['lender_markup'] > MARKUP_HINGE)
    .groupby('loan_type')['above_hinge'].agg(['size', 'mean'])
    .query('size >= 30')
)
lender_hinge_min = lender_above_share['mean'].min() if len(lender_above_share) else np.nan
lender_hinge_max = lender_above_share['mean'].max() if len(lender_above_share) else np.nan

# Compare one record per loan with one record per lender share.
# Exclude loans with conflicting target or markup values rather than choosing arbitrarily.
loan_level = train.assign(loan_markup=F_train['markup'].to_numpy()).groupby(
    'tbl_loan_id', as_index=False
).agg(
    target=('target', 'first'),
    target_nunique=('target', 'nunique'),
    loan_markup=('loan_markup', 'first'),
    markup_nunique=('loan_markup', 'nunique'),
    loan_type=('loan_type', 'first')
)
loan_level = loan_level[
    (loan_level['target_nunique'] == 1) &
    (loan_level['markup_nunique'] == 1)
].copy()

loan_defaulted = loan_level[loan_level['target'] == 1]
loan_repaid = loan_level[loan_level['target'] == 0]
lender_defaulted = lender_valid[lender_valid['target'] == 1]
lender_repaid = lender_valid[lender_valid['target'] == 0]

loan_hi_default = (loan_defaulted['loan_markup'] > MARKUP_HINGE).mean()
loan_hi_repaid = (loan_repaid['loan_markup'] > MARKUP_HINGE).mean()
lender_hi_default = (lender_defaulted['lender_markup'] > MARKUP_HINGE).mean()
lender_hi_repaid = (lender_repaid['lender_markup'] > MARKUP_HINGE).mean()

print(f"Whole-loan comparison uses {len(loan_level):,} distinct loans with consistent target and markup; "
      f"{len(lender_valid):,} of {len(train):,} lender-share rows have a defined lender markup. "
      f"Share above the {MARKUP_HINGE:.1%} hinge: whole-loan defaults {loan_hi_default:.1%} "
      f"vs repaid {loan_hi_repaid:.1%}; lender-share defaults {lender_hi_default:.1%} "
      f"vs repaid {lender_hi_repaid:.1%}.")
if len(lender_above_share):
    print(f"The product-level defaulted-loan share above the hinge ranges from "
          f"{lender_hinge_min:.1%} to {lender_hinge_max:.1%}; this supports markup as a "
          f"potential pricing signal at both loan and lender-share level, but it is not proof "
          f"of origination-time availability, so repayment-field timing must still be verified.")
else:
    print("There are too few defaulted observations per product to compare product-level lender markup reliably; "
          "the overall comparison is descriptive and repayment-field timing must still be verified.")


,target,median_lender_markup_repaid,median_lender_markup_defaulted,rows
loan_type,duration,,,
Type_1,7,0.7%,41.2%,"53,205"
Type_7,7,3.5%,8.3%,"2,790"
Type_5,14,5.6%,12.3%,"1,466"
Type_4,30,7.0%,20.6%,592
Type_6,7,2.0%,35.6%,352


Whole-loan comparison uses 66,518 distinct loans with consistent target and markup; 60,134 of 68,654 lender-share rows have a defined lender markup. Share above the 3.7% hinge: whole-loan defaults 97.2% vs repaid 7.9%; lender-share defaults 96.8% vs repaid 11.2%.
The product-level defaulted-loan share above the hinge ranges from 92.8% to 100.0%; this supports markup as a potential pricing signal at both loan and lender-share level, but it is not proof of origination-time availability, so repayment-field timing must still be verified.


**Interpretation:** The lender-side table reproduces the whole-loan pattern almost exactly: for every displayed product/term group, the median lender markup for repaid and defaulted loans matches the corresponding whole-loan median (for example, Type_1 over 7 days is 0.7% for repaid loans and 41.2% for defaults). 
This suggests that Lender_portion_to_be_repaid reflects the same underlying repayment and markup pattern as the whole-loan amount, but it does not establish when those amounts were recorded or whether they include post-default charges; therefore, repayment-derived features should remain outside the production model until their timing is confirmed.

## 4. Score the feature families
`Evaluator.run` applies the whole validation protocol from notebook 02: scheme B (with fold scores), C, D, E and the Ghana flag rate.
Each run takes about a minute.

In [7]:
ev = Evaluator(train, test, feats, folds)
B, P, H, U, M, K = (FAMILIES[k] for k in ['baseline', 'pricing', 'history', 'usd', 'macro', 'codes'])
PRODUCTION_BASE = ['log_amount_lcu', 'duration', 'is_new', 'lender_share']   # nothing derived from repayment amounts

FEATURE_SETS = {
    'M3 R0 competition: baseline (notebook 02)': B + K,
    'M3 R1 competition: R0 + pricing':      B + K + P,
    'M3 P0 production: no repayment fields': PRODUCTION_BASE + K,
    'M3 P2 production: P0 + all history':    PRODUCTION_BASE + K + H,
}
results = {}
for name, cols in FEATURE_SETS.items():
    results[name] = ev.run(name, cols)
    log_experiment(results[name])
pd.DataFrame(results.values()).set_index('experiment').drop(columns=['f1_B_folds']).round(3)

M3 R0 competition: baseline (notebook 02): F1 B 0.823 ± 0.020 | C 0.810 | D 0.565 | E AUC 0.929 | Ghana flag 11.5%


M3 R1 competition: R0 + pricing: F1 B 0.836 ± 0.012 | C 0.820 | D 0.578 | E AUC 0.949 | Ghana flag 10.5%


M3 P0 production: no repayment fields: F1 B 0.222 ± 0.030 | C 0.232 | D 0.153 | E AUC 0.654 | Ghana flag 15.0%


M3 P2 production: P0 + all history: F1 B 0.254 ± 0.026 | C 0.243 | D 0.124 | E AUC 0.673 | Ghana flag 15.7%


,n_features,f1_B,f1_B_sd,roc_auc_B,pr_auc_B,threshold_B,f1_C,f1_D,mean_pred_D,actual_D,auc_E_mean,auc_E_min,f1_E_weighted,flag_rate_Kenya,flag_rate_Ghana
experiment,,,,,,,,,,,,,,,
M3 R0 competition: baseline (notebook 02),7,0.823,0.020,0.993,0.903,0.50,0.810,0.565,0.058,0.095,0.929,0.840,0.772,0.017,0.115
M3 R1 competition: R0 + pricing,10,0.836,0.012,0.993,0.911,0.30,0.820,0.578,0.061,0.095,0.949,0.909,0.768,0.020,0.105
M3 P0 production: no repayment fields,6,0.222,0.030,0.798,0.185,0.12,0.232,0.153,0.064,0.095,0.654,0.564,0.222,0.022,0.150
M3 P2 production: P0 + all history,12,0.254,0.026,0.861,0.201,0.12,0.243,0.124,0.064,0.095,0.673,0.578,0.252,0.025,0.157


In [8]:
r0, r1 = results['M3 R0 competition: baseline (notebook 02)'], results['M3 R1 competition: R0 + pricing']
f0 = np.array(r0['f1_B_folds'].split(), float); f1 = np.array(r1['f1_B_folds'].split(), float)
print(f"Pricing: scheme B {r0['f1_B']:.3f} -> {r1['f1_B']:.3f} (+{r1['f1_B'] - r0['f1_B']:.3f}); "
      f"better in {(f1 > f0).sum()} of 5 folds; fold sd {r0['f1_B_sd']:.3f} -> {r1['f1_B_sd']:.3f}.")

Pricing: scheme B 0.823 -> 0.836 (+0.013); better in 4 of 5 folds; fold sd 0.020 -> 0.012.


In [9]:
# Production track: low F1, but how well does it RANK? P2 was the last run, so ev.last_oof_B holds its scores.
p2 = results['M3 P2 production: P0 + all history']
oof = ev.last_oof_B
top = oof >= np.quantile(oof, 0.9)
thr = p2['threshold_B']
print(f"P2 ROC AUC {p2['roc_auc_B']:.3f}; at threshold {thr:.2f}: precision {y[oof >= thr].mean():.1%}, "
      f"recall {(oof >= thr)[y == 1].mean():.1%}.")
print(f"The riskiest 10% of loans by P2 score contain {y[top].sum() / y.sum():.0%} of all defaults "
      f"(a random 10% would contain 10%).")

P2 ROC AUC 0.861; at threshold 0.12: precision 21.4%, recall 31.3%.
The riskiest 10% of loans by P2 score contain 57% of all defaults (a random 10% would contain 10%).


In [10]:
# Run the remaining competition and production feature families.
def unique_cols(cols):
    return list(dict.fromkeys(cols))

R1 = FEATURE_SETS['M3 R1 competition: R0 + pricing']
R4 = unique_cols([c for c in R1 if c != 'log_amount_lcu'] + U)
P0 = FEATURE_SETS['M3 P0 production: no repayment fields']
P2 = FEATURE_SETS['M3 P2 production: P0 + all history']

EXTRA_FEATURE_SETS = {
    'M3 R2 competition: R1 + label-free history':
        unique_cols(R1 + H[:3]),
    'M3 R3 competition: R1 + all history':
        unique_cols(R1 + H),
    'M3 R4 competition: R1 with USD amount':
        R4,
    'M3 R5 competition: R4 + macro':
        unique_cols(R4 + M),
    'M3 R6 competition: R4 without codes':
        [c for c in R4 if c not in K],
    'M3 P1 production: P0 + label-free history':
        unique_cols(P0 + H[:3]),
    'M3 P3 production: P2 + macro':
        unique_cols(P2 + M),
    'M3 P4 production: P2 with USD amount':
        unique_cols([c for c in P2 if c != 'log_amount_lcu'] + U),
}

# Each challenger is compared with the nearest simpler parent model.
PARENTS = {
    'M3 R2 competition: R1 + label-free history': 'M3 R1 competition: R0 + pricing',
    'M3 R3 competition: R1 + all history': 'M3 R1 competition: R0 + pricing',
    'M3 R4 competition: R1 with USD amount': 'M3 R1 competition: R0 + pricing',
    'M3 R5 competition: R4 + macro': 'M3 R4 competition: R1 with USD amount',
    'M3 R6 competition: R4 without codes': 'M3 R4 competition: R1 with USD amount',
    'M3 P1 production: P0 + label-free history': 'M3 P0 production: no repayment fields',
    'M3 P3 production: P2 + macro': 'M3 P2 production: P0 + all history',
    'M3 P4 production: P2 with USD amount': 'M3 P2 production: P0 + all history',
}

# Capture P1's out-of-fold Scheme B scores immediately after its run.
# Evaluator.last_oof_B is overwritten by each subsequent experiment.
p1_oof = None
for name, cols in EXTRA_FEATURE_SETS.items():
    results[name] = ev.run(name, cols)
    if name == 'M3 P1 production: P0 + label-free history':
        p1_oof = ev.last_oof_B.copy()
    log_experiment(results[name])

if p1_oof is None:
    raise RuntimeError("P1 out-of-fold scores were not captured.")
if len(p1_oof) != len(y):
    raise ValueError("P1 out-of-fold scores are not aligned with the training labels.")

def _fold_values(result):
    value = result.get('f1_B_folds', '')
    if isinstance(value, str):
        return np.fromstring(value, sep=' ')
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.array([], dtype=float)

def _guardrail_value(result, scheme):
    # Scheme E is reported by Evaluator as f1_E_weighted (not f1_E).
    # Keep a fallback for older evaluator versions that expose f1_E.
    key_candidates = {
        'C': ['f1_C'],
        'D': ['f1_D'],
        'E': ['f1_E_weighted', 'f1_E'],
    }[scheme]
    for key in key_candidates:
        value = result.get(key, np.nan)
        try:
            numeric = float(value)
            if np.isfinite(numeric):
                return numeric
        except (TypeError, ValueError):
            pass
    return np.nan

GUARDRAIL_TOLERANCE = 0.01  # one F1 point: practical tolerance for C/D/E movement
protocol_rows = []
for name, cols in EXTRA_FEATURE_SETS.items():
    candidate = results[name]
    parent = results[PARENTS[name]]
    candidate_folds = _fold_values(candidate)
    parent_folds = _fold_values(parent)
    n_common = min(len(candidate_folds), len(parent_folds))
    fold_wins = int((candidate_folds[:n_common] > parent_folds[:n_common]).sum())
    b_gain = float(candidate['f1_B']) - float(parent['f1_B'])
    b_sd = float(parent.get('f1_B_sd', np.nan))
    meaningful_B = (b_gain > b_sd) or (n_common >= 5 and fold_wins >= 4)

    guardrail_deltas = {
        s: _guardrail_value(candidate, s) - _guardrail_value(parent, s)
        for s in ['C', 'D', 'E']
    }
    available_guardrails = [v for v in guardrail_deltas.values() if np.isfinite(v)]
    no_clear_guardrail_harm = (
        len(available_guardrails) == 3 and
        all(v >= -GUARDRAIL_TOLERANCE for v in available_guardrails)
    )
    passes = meaningful_B and no_clear_guardrail_harm
    protocol_rows.append({
        'experiment': name, 'parent': PARENTS[name],
        'f1_B': candidate['f1_B'], 'delta_f1_B': b_gain,
        'fold_wins': f'{fold_wins}/{n_common}',
        'f1_C_delta': guardrail_deltas['C'],
        'f1_D_delta': guardrail_deltas['D'],
        'f1_E_delta': guardrail_deltas['E'],
        'protocol': 'PASS' if passes else 'REVIEW'
    })
    guardrail_text = ', '.join(
        f"{s} {delta:+.3f}" if np.isfinite(delta) else f"{s} unavailable"
        for s, delta in guardrail_deltas.items()
    )
    print(f"{name}: Scheme B F1 {parent['f1_B']:.3f} -> {candidate['f1_B']:.3f} "
          f"({b_gain:+.3f}); fold wins {fold_wins}/{n_common}; "
          f"{guardrail_text}. "
          f"{'Passes' if passes else 'Does not yet pass'} the protocol "
          f"(B gain > parent fold SD {b_sd:.3f} or >=4/5 fold wins, "
          f"and no C/D/E decline beyond {GUARDRAIL_TOLERANCE:.3f}).")

protocol_summary = pd.DataFrame(protocol_rows).set_index('experiment')
display(protocol_summary.round(3))

M3 R2 competition: R1 + label-free history: F1 B 0.825 ± 0.012 | C 0.826 | D 0.562 | E AUC 0.934 | Ghana flag 10.8%


M3 R3 competition: R1 + all history: F1 B 0.832 ± 0.014 | C 0.829 | D 0.534 | E AUC 0.927 | Ghana flag 11.3%


M3 R4 competition: R1 with USD amount: F1 B 0.828 ± 0.009 | C 0.816 | D 0.471 | E AUC 0.934 | Ghana flag 9.6%


M3 R5 competition: R4 + macro: F1 B 0.822 ± 0.009 | C 0.820 | D 0.471 | E AUC 0.939 | Ghana flag 10.5%


M3 R6 competition: R4 without codes: F1 B 0.826 ± 0.009 | C 0.816 | D 0.503 | E AUC 0.937 | Ghana flag 10.6%


M3 P1 production: P0 + label-free history: F1 B 0.243 ± 0.035 | C 0.227 | D 0.145 | E AUC 0.663 | Ghana flag 15.1%


M3 P3 production: P2 + macro: F1 B 0.262 ± 0.035 | C 0.244 | D 0.131 | E AUC 0.675 | Ghana flag 15.4%


M3 P4 production: P2 with USD amount: F1 B 0.244 ± 0.023 | C 0.237 | D 0.144 | E AUC 0.678 | Ghana flag 15.7%
M3 R2 competition: R1 + label-free history: Scheme B F1 0.836 -> 0.825 (-0.011); fold wins 1/5; C +0.006, D -0.017, E -0.025. Does not yet pass the protocol (B gain > parent fold SD 0.012 or >=4/5 fold wins, and no C/D/E decline beyond 0.010).
M3 R3 competition: R1 + all history: Scheme B F1 0.836 -> 0.832 (-0.004); fold wins 2/5; C +0.009, D -0.044, E -0.022. Does not yet pass the protocol (B gain > parent fold SD 0.012 or >=4/5 fold wins, and no C/D/E decline beyond 0.010).
M3 R4 competition: R1 with USD amount: Scheme B F1 0.836 -> 0.828 (-0.008); fold wins 1/5; C -0.003, D -0.107, E +0.005. Does not yet pass the protocol (B gain > parent fold SD 0.012 or >=4/5 fold wins, and no C/D/E decline beyond 0.010).
M3 R5 competition: R4 + macro: Scheme B F1 0.828 -> 0.822 (-0.006); fold wins 0/5; C +0.004, D -0.001, E +0.007. Does not yet pass the protocol (B gain > parent fold SD 0

,parent,f1_B,delta_f1_B,fold_wins,f1_C_delta,f1_D_delta,f1_E_delta,protocol
experiment,,,,,,,,
M3 R2 competition: R1 + label-free history,M3 R1 competition: R0 + pricing,0.825,-0.011,1/5,0.006,-0.017,-0.025,REVIEW
M3 R3 competition: R1 + all history,M3 R1 competition: R0 + pricing,0.832,-0.004,2/5,0.009,-0.044,-0.022,REVIEW
M3 R4 competition: R1 with USD amount,M3 R1 competition: R0 + pricing,0.828,-0.008,1/5,-0.003,-0.107,0.005,REVIEW
M3 R5 competition: R4 + macro,M3 R4 competition: R1 with USD amount,0.822,-0.006,0/5,0.004,-0.001,0.007,REVIEW
M3 R6 competition: R4 without codes,M3 R4 competition: R1 with USD amount,0.826,-0.002,3/5,0.000,0.032,-0.044,REVIEW
M3 P1 production: P0 + label-free history,M3 P0 production: no repayment fields,0.243,0.022,4/5,-0.004,-0.008,-0.005,PASS
M3 P3 production: P2 + macro,M3 P2 production: P0 + all history,0.262,0.008,3/5,0.001,0.007,0.044,REVIEW
M3 P4 production: P2 with USD amount,M3 P2 production: P0 + all history,0.244,-0.009,1/5,-0.006,0.020,0.044,REVIEW


## 5. How far does Ghana sit outside the training range?

In [11]:
rng_rows = []
ke = (test['country_id'] == 'Kenya').to_numpy()
for c in ['log_amount_usd', 'duration', 'markup', 'log_amount_lcu', 'lender_share', 'daily_rate']:
    p1, p99 = F_train[c].quantile([0.01, 0.99])
    out = lambda s: ((s < p1) | (s > p99)).mean()
    rng_rows.append({'feature': c, 'ghana_outside_p1_p99': out(F_test[c][gh]), 'kenya_test_outside_p1_p99': out(F_test[c][ke])})
out_of_range = pd.DataFrame(rng_rows).sort_values('ghana_outside_p1_p99', ascending=False)
display(out_of_range.style.format({'ghana_outside_p1_p99': '{:.1%}', 'kenya_test_outside_p1_p99': '{:.1%}'}))

,feature,ghana_outside_p1_p99,kenya_test_outside_p1_p99
0,log_amount_usd,23.2%,1.8%
1,duration,13.8%,0.9%
2,markup,8.4%,1.2%
3,log_amount_lcu,3.2%,1.8%
4,lender_share,1.5%,1.0%
5,daily_rate,0.5%,1.2%


## 6. Sensitivity tests

In [12]:
# Zero-funded rows: are they also in test?
zf_train = (F_train['is_zero_funded'] == 1).to_numpy()
zf_test = (F_test['is_zero_funded'] == 1).to_numpy()
print(f"Zero-funded: {zf_train.sum():,} train rows (default rate {y[zf_train].mean():.1%}); "
      f"{zf_test[ke].sum():,} Kenya test rows ({zf_test[ke].mean():.1%}); {zf_test[gh].sum()} Ghana test rows.")
s1 = ev.run('M3 S1 R1, zero-funded rows dropped from training', FEATURE_SETS['M3 R1 competition: R0 + pricing'],
            train_mask=~zf_train)
log_experiment(s1)

Zero-funded: 8,520 train rows (default rate 0.0%); 1,868 Kenya test rows (12.4%); 0 Ghana test rows.


M3 S1 R1, zero-funded rows dropped from training: F1 B 0.807 ± 0.026 | C 0.792 | D 0.625 | E AUC 0.941 | Ghana flag 10.4%


,date,experiment,f1_B,f1_B_sd,pr_auc_B,f1_C,f1_D,threshold_B,n_features,roc_auc_B,f1_B_folds,mean_pred_D,actual_D,auc_E_mean,auc_E_min,f1_E_weighted,flag_rate_Kenya,flag_rate_Ghana
0,2026-11-24,M2 baseline: 5 numeric + lender_id + loan_type...,0.823377,0.019544,0.903010,0.810417,0.564841,0.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-11-24,"M2 starter-style LogisticRegression, no IDs, S...",0.683145,0.016401,0.715992,0.681250,0.396456,0.84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-11-24,M3 R0 competition: baseline (notebook 02),0.823377,0.019544,0.903010,0.810417,0.564841,0.50,7.0,0.993023,0.833 0.811 0.793 0.850 0.830,0.058002,0.095249,0.929358,0.840093,0.771686,0.017453,0.115461
3,2026-11-24,M3 R1 competition: R0 + pricing,0.836160,0.011937,0.911404,0.819845,0.578249,0.30,10.0,0.993341,0.841 0.835 0.821 0.828 0.856,0.060997,0.095249,0.949261,0.909207,0.767941,0.019709,0.104681
4,2026-11-24,M3 P0 production: no repayment fields,0.221717,0.029834,0.184756,0.231600,0.152632,0.12,6.0,0.798291,0.208 0.213 0.252 0.178 0.259,0.064207,0.095249,0.653807,0.563907,0.221694,0.022098,0.150355
5,2026-11-24,M3 P2 production: P0 + all history,0.253933,0.025586,0.201272,0.243124,0.124260,0.12,12.0,0.860681,0.235 0.244 0.250 0.237 0.304,0.063556,0.095249,0.673100,0.578109,0.252158,0.024686,0.156879
6,2026-11-24,M3 R2 competition: R1 + label-free history,0.825301,0.012036,0.912041,0.825977,0.561514,0.26,13.0,0.993660,0.825 0.817 0.807 0.839 0.837,0.048850,0.095249,0.934155,0.846503,0.742509,0.019908,0.108369
7,2026-11-24,M3 R3 competition: R1 + all history,0.831881,0.014078,0.912234,0.828746,0.534375,0.26,16.0,0.993399,0.820 0.818 0.825 0.841 0.855,0.049929,0.095249,0.927363,0.817483,0.745520,0.019577,0.113475
8,2026-11-24,M3 R4 competition: R1 with USD amount,0.828064,0.008744,0.910855,0.816405,0.471098,0.46,10.0,0.993177,0.830 0.832 0.822 0.815 0.841,0.053838,0.095249,0.933679,0.856876,0.772710,0.017586,0.095603
9,2026-11-24,M3 R5 competition: R4 + macro,0.822398,0.009111,0.911235,0.820221,0.470588,0.36,13.0,0.993583,0.823 0.825 0.816 0.811 0.837,0.054354,0.095249,0.938536,0.882751,0.779939,0.018648,0.104681


In [13]:
# Assumption 3: test a 30-day label-availability lag for customer history.
# Assumption 4: test same-year macro values as a deliberate look-ahead sensitivity.
# The production P2/P3 feature definitions and the fixed notebook 02 folds are retained.

p2_cols = FEATURE_SETS['M3 P2 production: P0 + all history']
p2_reference = results['M3 P2 production: P0 + all history']

# Assumption 3: a prior default is considered known only 30 days after its due date.
feats_lag30 = build_features(train, test, econ, label_lag_days=30, macro_lag_years=1)
ev_lag30 = Evaluator(train, test, feats_lag30, folds)
lag30_result = ev_lag30.run('M3 S2 P2, customer-history labels known 30 days after due date',
                            p2_cols)
log_experiment(lag30_result)

lag30_delta_B = float(lag30_result['f1_B']) - float(p2_reference['f1_B'])
lag30_delta_C = float(lag30_result.get('f1_C', np.nan)) - float(p2_reference.get('f1_C', np.nan))
lag30_delta_D = float(lag30_result.get('f1_D', np.nan)) - float(p2_reference.get('f1_D', np.nan))
print(f"30-day label lag: P2 Scheme B F1 "
      f"{p2_reference['f1_B']:.3f} -> {lag30_result['f1_B']:.3f} "
      f"({lag30_delta_B:+.3f}; reference Scheme B fold SD "
      f"{p2_reference['f1_B_sd']:.3f}); C change {lag30_delta_C:+.3f}; "
      f"D change {lag30_delta_D:+.3f}.")
if abs(lag30_delta_B) <= float(p2_reference['f1_B_sd']):
    lag30_status = "Robust within the observed Scheme B fold-to-fold variation."
else:
    lag30_status = "Sensitive beyond the observed Scheme B fold-to-fold variation."
print(f"Assumption 3 status: {lag30_status}")

# Assumption 4: same-year macro values are a sensitivity check only. They may not
# have been published by the loan's disbursement date, so they are not the
# default production choice without a publication-date guarantee.
feats_sameyear = build_features(train, test, econ, label_lag_days=0, macro_lag_years=0)
ev_sameyear = Evaluator(train, test, feats_sameyear, folds)
p3_cols = EXTRA_FEATURE_SETS['M3 P3 production: P2 + macro']
p3_reference = results['M3 P3 production: P2 + macro']
sameyear_result = ev_sameyear.run('M3 S3 P3, same-year macro sensitivity (look-ahead check)',
                                  p3_cols)
log_experiment(sameyear_result)

sameyear_delta_B = float(sameyear_result['f1_B']) - float(p3_reference['f1_B'])
sameyear_delta_C = float(sameyear_result.get('f1_C', np.nan)) - float(p3_reference.get('f1_C', np.nan))
sameyear_delta_D = float(sameyear_result.get('f1_D', np.nan)) - float(p3_reference.get('f1_D', np.nan))
print(f"Same-year macro sensitivity: P3 Scheme B F1 "
      f"{p3_reference['f1_B']:.3f} -> {sameyear_result['f1_B']:.3f} "
      f"({sameyear_delta_B:+.3f}; reference Scheme B fold SD "
      f"{p3_reference['f1_B_sd']:.3f}); C change {sameyear_delta_C:+.3f}; "
      f"D change {sameyear_delta_D:+.3f}.")
if abs(sameyear_delta_B) <= float(p3_reference['f1_B_sd']):
    sameyear_status = ("No material Scheme B change beyond fold variation; "
                       "retain the previous-year join for timing integrity.")
else:
    sameyear_status = ("Scheme B is sensitive to same-year macro values; "
                       "retain the previous-year join unless publication timing is proven.")
print(f"Assumption 4 status: {sameyear_status}")

M3 S2 P2, customer-history labels known 30 days after due date: F1 B 0.246 ± 0.033 | C 0.237 | D 0.139 | E AUC 0.666 | Ghana flag 15.1%
30-day label lag: P2 Scheme B F1 0.254 -> 0.246 (-0.008; reference Scheme B fold SD 0.026); C change -0.006; D change +0.015.
Assumption 3 status: Robust within the observed Scheme B fold-to-fold variation.


M3 S3 P3, same-year macro sensitivity (look-ahead check): F1 B 0.250 ± 0.039 | C 0.249 | D 0.131 | E AUC 0.653 | Ghana flag 13.6%
Same-year macro sensitivity: P3 Scheme B F1 0.262 -> 0.250 (-0.012; reference Scheme B fold SD 0.035); C change +0.004; D change +0.000.
Assumption 4 status: No material Scheme B change beyond fold variation; retain the previous-year join for timing integrity.


## 7. Save the selected feature sets

In [14]:
selected = {
    'competition': FEATURE_SETS['M3 R1 competition: R0 + pricing'],
    'production':  EXTRA_FEATURE_SETS['M3 P1 production: P0 + label-free history'],   # P1 passed the protocol; P2 failed scheme D
}
(OUT / 'feature_sets.json').write_text(json.dumps(selected, indent=2))
print(json.dumps(selected, indent=2))

{
  "competition": [
    "log_amount_lcu",
    "markup",
    "duration",
    "is_new",
    "lender_share",
    "lender_id",
    "loan_type",
    "daily_rate",
    "markup_high",
    "markup_excess"
  ],
  "production": [
    "log_amount_lcu",
    "duration",
    "is_new",
    "lender_share",
    "lender_id",
    "loan_type",
    "hist_n_prior",
    "hist_days_since_last",
    "hist_log_amount_vs_prior"
  ]
}


## 8. Assumptions log

| # | Assumption | Evidence from this run | Status / decision |
| --- | --- | --- | --- |
| 1 | Kenya rows can be reweighted to represent Ghana | Notebook 02 adversarial validation returned AUC = 1.00. Notebook 03 does not supply Ghana labels or prove common support. | **Not established.** Do not claim Kenya reweighting or a Kenya-derived threshold is validated for Ghana. Re-test overlap after currency-neutral and product-aware features; obtain labelled Ghana outcomes if overlap remains absent. |
| 2 | `Total_Amount_to_Repay` is fixed at signing | Loan-level and lender-share markup comparisons are descriptive. The data does not establish when repayment fields were recorded or whether they were later updated. | **Conditional.** Keep markup features in the competition track. Exclude repayment-derived fields from the production track until origination-time availability is confirmed. |
| 3 | A prior default is known on its due date | P2 Scheme B F1 changed from 0.254 to 0.246 under a 30-day recognition lag (delta -0.008; reference fold SD 0.026). Scheme C changed -0.006 and Scheme D +0.015. | **Robust within observed fold variation**, but the operational default-recognition rule still needs confirmation. |
| 4 | Macro joins use the previous year | P3 Scheme B F1 changed from 0.262 to 0.250 under same-year macro values (delta -0.012; reference fold SD 0.035). Scheme C changed +0.004 and Scheme D approximately 0.000. | **No material Scheme B change beyond fold variation.** Retain previous-year values for publication-timing integrity. |
| 5 | Zero-funded rows are genuine repaid loans | 8,520 training rows have zero lender funding and 0% observed defaults; 1,868 Kenya test rows are also zero-funded, while Ghana has none. Dropping these rows changes R1 Scheme B F1 from 0.836 to 0.807 (-0.029), Scheme C from 0.820 to 0.792 (-0.028), and Scheme D from 0.578 to 0.625 (+0.047). | **Unresolved data-quality / population-shift risk.** Keep the sensitivity visible, but do not treat zero funding as a reliable risk signal until its business meaning and label-generation process are confirmed. |
| 6 | `target` is consistent across all lender shares of a loan | Notebook 01 identified two loan IDs with conflicting target values. | **Known exception.** Exclude inconsistent loans from loan-level summaries or resolve them from source records; do not silently select a label. |
| 7 | Missing Ghana macro values can be left missing | Notebook 01 found gaps in selected Ghana macro indicators. | Preserve missingness and monitor feature coverage by country. Do not invent Ghana macro values. |

**Feature-selection result from this run:** R1 (competition: baseline plus pricing) improved Scheme B F1 from 0.823 to 0.836 and won 4/5 folds. The production track is materially weaker: P0 scored 0.222 and P2 scored 0.254, while P2 Scheme D F1 fell from 0.153 to 0.124. P1 (P0 plus label-free history) passes the protocol: Scheme B F1 0.243 (+0.022 vs P0), 4/5 fold wins, and changes of -0.004 in C, -0.008 in D and -0.005 in weighted E, all within the 0.010 tolerance. P1 is the production set from here on.

**Ghana transfer limitation:** An adversarial AUC of 1.00 means the tested Ghana and Kenya rows were perfectly separable in that feature space. This is evidence of no demonstrated common support in the tested representation, not proof that no transformed representation can overlap. Currency conversion and feature engineering do not create labelled Ghana outcomes. Do not claim that a Kenya-derived threshold is validated for Ghana.


## 9. Sensitivity: keep zero-funded rows and add a flag
Same R1 protocol, with `is_zero_funded` added as a feature.

In [15]:
zero_flag_cols = FEATURE_SETS['M3 R1 competition: R0 + pricing'] + ['is_zero_funded']
zero_flag_result = ev.run('M3 S1b R1, zero-funded flag added', zero_flag_cols, verbose=False)
log_experiment(zero_flag_result)
r1_result = results['M3 R1 competition: R0 + pricing']
print(f"Zero-funded flag: scheme B F1 {r1_result['f1_B']:.3f} -> {zero_flag_result['f1_B']:.3f}, "
      f"scheme D {r1_result['f1_D']:.3f} -> {zero_flag_result['f1_D']:.3f}, held-out-lender AUC "
      f"{r1_result['auc_E_mean']:.3f} -> {zero_flag_result['auc_E_mean']:.3f}. The flag does not improve scheme B "
      "or lender transfer and may encode a reporting pattern rather than borrower risk, so the rows stay in "
      "and the flag stays out.")

Zero-funded flag: scheme B F1 0.836 -> 0.830, scheme D 0.578 -> 0.607, held-out-lender AUC 0.949 -> 0.932. The flag does not improve scheme B or lender transfer and may encode a reporting pattern rather than borrower risk, so the rows stay in and the flag stays out.


## 10. Tag this notebook's log rows as protocol v1

In [16]:
# Rows written by this notebook (and earlier modules) used the v1 evaluator.
# Rows that already carry a protocol (e.g. v2-nested from 03b) are left untouched.
log = pd.read_csv(OUT / 'experiments.csv')
if 'protocol' not in log.columns:
    log['protocol'] = np.nan
log['protocol'] = log['protocol'].fillna('v1')
log.to_csv(OUT / 'experiments.csv', index=False)
print(log['protocol'].value_counts().to_string())

protocol
v1    18
